[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyMongo and Beanie, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)

# Query Operators


## What you will be able to do

Write filters that match what you meant. Say what a nested dictionary in a filter actually asks for,
which is not what almost anybody expects, and reach for dot notation instead. Use `$gt`, `$in`,
`$nin`, `$exists` and `$type`, and put them at the level where they belong. Say why two conditions
on the same array can both be satisfied by a document that has no single element satisfying both,
and fix it with `$elemMatch`. And know what `{"field": None}` matches, which is more than you think.


## The idea

### The problem

A filter is a document, and so is the thing you are filtering. That symmetry is elegant and it hides
a decision: when you write `{"size": {"w": 21}}`, is the inner dictionary a description of what
`size` should look like, or a set of conditions on it?

It is the first. `{"size": {"w": 21}}` asks for documents whose `size` **is exactly** `{"w": 21}`,
one key, that value, and nothing else. Almost nobody means that.

### What dot notation is

A path into a document, written as a string: `"size.w"` reaches the `w` inside `size`, and
`"entries.n"` reaches the `n` inside every element of the `entries` array. A field name with a dot
in it is how MongoDB says "go inside", and it works in filters, projections, sorts and indexes.

### Why a nested dictionary is not conditions

Because a dictionary of conditions would be ambiguous with a dictionary of values, and MongoDB chose
values. The way to write conditions is with operators, which are keys beginning with `$`, and they
go **inside** the field: `{"price": {"$gt": 100}}`, never `{"$gt": 100}`.

### Where this shows up

Every non-trivial query. The array rules in particular decide whether a reporting query is right or
merely plausible, and a plausible one is worse than a broken one.

### What this notebook covers

Whole-document equality against dot notation. Operators and where they go. `$in`, `$nin`, `$exists`,
`$type`, and what `None` means. Arrays: why conditions spread across elements, and `$elemMatch`.
Then the four failures, two of which return a confident wrong answer.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import pymongo

client = pymongo.MongoClient("mongodb://127.0.0.1:27017/shop", tz_aware=True)
shop = client.get_default_database()

print("with dot notation:      ", shop.products.count_documents({"size.w": 21}))
print("as a nested dictionary: ", shop.products.count_documents({"size": {"w": 21}}))

exact = shop.products.find_one({"_id": 0})["size"]
print("the whole subdocument:  ", shop.products.count_documents({"size": exact}), "for", exact)
print("the same keys, reordered:", shop.products.count_documents(
    {"size": {"h": exact["h"], "w": exact["w"]}}))
client.close()
```

```
with dot notation:       9
as a nested dictionary:  0
the whole subdocument:   2 for {'w': 21, 'h': 34}
the same keys, reordered: 0
```

Four filters over the same five hundred documents. The second matched nothing, because no `size` is
exactly `{"w": 21}`. The third matched, because that **is** the whole subdocument. And the fourth
matched nothing again, because whole-document equality compares the fields in order, so swapping two
keys asks a different question.


## Setup

Seven imports, MongoDB, the boot cell, and three helpers.

- `pymongo` is the driver, and `time` is there for the boot cell's readiness loop
- `subprocess` and `os` install and start the server, `sys` names this Python
- `random` seeds the data the same way every run, with `version` and `PackageNotFoundError`

`ids` gives the sorted `_id`s a query matched, which is how most of this notebook compares two
filters. `build_scores` makes two tiny collections: `scores`, whose two documents hold the same
values arranged differently inside an array, and `maybe`, which has a null, a missing field and a
real value. `failed` prints an `OperationFailure`'s message without the cluster time.


In [1]:
import os
import random
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("pymongo") != "4.18.1" or version("beanie") != "2.2.0":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "pymongo==4.18.1", "beanie==2.2.0"], check=True)

import pymongo

DBPATH = "/content/mongo" if os.path.isdir("/content") else "/tmp/guide_mongo/rs"
LOGPATH = f"{DBPATH}.log"
URI = "mongodb://127.0.0.1:27017/shop"                              # no credential, anywhere
PUBLISHED = ["jammy", "noble"]                                      # codenames MongoDB builds for


def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(timeout=2000):
    """Whether a mongod is there, asked directly rather than through topology discovery."""
    try:
        with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                                 serverSelectionTimeoutMS=timeout) as client:
            client.admin.command("ping")
            return True
    except pymongo.errors.PyMongoError:
        return False


def install_server():
    """Add MongoDB's own apt repository and install the server package. Linux only."""
    if shell("which mongod")[0] == 0:
        return "already installed"

    codename = shell("lsb_release -cs")[1]
    if codename not in PUBLISHED:                                   # an unpublished one breaks apt
        print(f"  Ubuntu '{codename}' has no MongoDB repository; using '{PUBLISHED[-1]}' instead")
        codename = PUBLISHED[-1]

    if not shell("grep -o avx /proc/cpuinfo | head -1")[1]:
        raise RuntimeError("This CPU has no AVX. Every MongoDB build since 5.0 needs it, so "
                           "neither the apt package nor the tarball will start here.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"curl -fsSL https://www.mongodb.org/static/pgp/server-8.0.asc "
          f"| {sudo}gpg --dearmor -o /usr/share/keyrings/mongodb-8.0.gpg")
    shell(f'echo "deb [signed-by=/usr/share/keyrings/mongodb-8.0.gpg] '
          f'https://repo.mongodb.org/apt/ubuntu {codename}/mongodb-org/8.0 multiverse" '
          f'| {sudo}tee /etc/apt/sources.list.d/mongodb-8.0.list')
    shell(f"{sudo}apt-get -qq update "                              # this one list file only
          f"-o Dir::Etc::sourcelist=sources.list.d/mongodb-8.0.list "
          f"-o Dir::Etc::sourceparts=-")
    code, out = shell(f"{sudo}apt-get -qq -y install mongodb-org-server")
    if shell("which mongod")[0] != 0:
        raise RuntimeError(f"mongodb-org-server did not install. apt said: {out[-400:]}")
    return f"installed from the {codename} repository"


def start_server(wait=30):
    """Start mongod with a replica set name, idempotently. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No mongod is answering on 127.0.0.1:27017. Start your own server "
                           "with --replSet rs0 and run this again: this cell only installs one "
                           "on Linux, which is what Colab runs.")

    print(" ", install_server())
    os.makedirs(DBPATH, exist_ok=True)
    code, out = shell(f"mongod --dbpath {DBPATH} --replSet rs0 --bind_ip 127.0.0.1 "
                      f"--fork --logpath {LOGPATH}")
    if code != 0:                                                   # --fork hides the reason
        print("  mongod did not start. The last lines of its log:")
        print("   ", shell(f"tail -20 {LOGPATH}")[1].replace("\n", "\n    "))
        raise RuntimeError("mongod exited. The log above says why.")

    for attempt in range(1, wait + 1):
        if answering():
            return "installed and started"
        print(f"  waiting for mongod ({attempt})")
        time.sleep(1)
    raise RuntimeError(f"mongod did not answer within {wait} seconds.")

def initiate(wait=30):
    """Make the single node a replica set, which is what transactions and migrations need."""
    with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                             serverSelectionTimeoutMS=2000) as boot:
        try:                                                        # an explicit host, not getHostName()
            boot.admin.command("replSetInitiate",
                               {"_id": "rs0", "members": [{"_id": 0, "host": "127.0.0.1:27017"}]})
        except pymongo.errors.OperationFailure as error:
            if error.code != 23:                                    # 23 is AlreadyInitialized
                raise

        for attempt in range(1, wait + 1):
            hello = boot.admin.command("hello")
            if hello.get("isWritablePrimary"):
                return f"replica set {hello['setName']}, primary"
            time.sleep(1)
    raise RuntimeError(f"No primary after {wait} seconds. The last hello was: {hello}")

SIZE = 500                                                          # Indexes and the catalog raise this
KINDS = ["laptop", "monitor", "keyboard", "mouse", "cable"]
MAKERS = ["Aster", "Belden", "Corvid", "Dalgo"]


def seed(size=None, force=False):
    """Fill shop.products and shop.reviews, once, from a fixed seed so every run agrees."""
    size = SIZE if size is None else size
    client = pymongo.MongoClient(URI, tz_aware=True)
    shop = client.get_default_database()

    if not force and shop.products.estimated_document_count() == size:
        client.close()
        return size

    shop.products.drop()
    shop.reviews.drop()
    random.seed(0)                                                  # the whole reason runs agree

    products, reviews = [], []
    for number in range(size):
        kind = KINDS[number % len(KINDS)]
        product = {
            "_id": number,
            "sku": f"{kind[:3].upper()}-{number:06d}",
            "name": f"{MAKERS[number % len(MAKERS)]} {kind} {number}",
            "maker": MAKERS[number % len(MAKERS)],
            "kind": kind,
            "price": round(random.uniform(5, 2000), 2),
            "stock": random.randint(0, 400),
            "tags": sorted(random.sample(["sale", "new", "refurbished", "bulk", "clearance"], 2)),
            "size": {"w": random.randint(5, 60), "h": random.randint(2, 40)},
        }
        products.append(product)
        for _ in range(random.randint(0, 3)):
            reviews.append({"product_id": number, "stars": random.randint(1, 5),
                            "body": f"A review of {product['name']}"})

    for start in range(0, len(products), 5000):                     # batches, not one huge insert
        shop.products.insert_many(products[start:start + 5000])
    for start in range(0, len(reviews), 5000):
        shop.reviews.insert_many(reviews[start:start + 5000])

    client.close()
    return size


def report():
    """One line naming what this notebook is running against."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        build = client.admin.command("buildInfo")["version"].split(".")[0]
        shop = client.get_default_database()
        return (f"MongoDB {build} | pymongo {version('pymongo')} | beanie {version('beanie')} "
                f"| products: {shop.products.count_documents({})}")

def failed(error):
    """An OperationFailure's real message, without the cluster time that changes every run."""
    return f"{type(error).__name__}: {error.details.get('errmsg', error)}"


def ids(cursor):
    """The _ids a query matched, sorted, which is what most of this notebook is comparing."""
    return sorted(document["_id"] for document in cursor)


def build_scores():
    """Two documents whose arrays hold the same values in different elements."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        shop = client.get_default_database()
        shop.scores.drop()
        shop.scores.insert_many([
            {"_id": 1, "entries": [{"kind": "a", "n": 5}, {"kind": "b", "n": 1}]},
            {"_id": 2, "entries": [{"kind": "a", "n": 1}, {"kind": "b", "n": 5}]},
        ])
        shop.maybe.drop()
        shop.maybe.insert_many([{"_id": 1, "v": None}, {"_id": 2}, {"_id": 3, "v": 7}])
        return "scores and maybe are ready"


print("server: ", start_server())
print("replica:", initiate())
print("seeded: ", seed(), "products")
print(build_scores())
print(report())


server:  already running
replica: replica set rs0, primary
seeded:  500 products
scores and maybe are ready
MongoDB 8 | pymongo 4.18.1 | beanie 2.2.0 | products: 500


## Worked examples

### A nested dictionary is a value, not a question

The rule in one cell:


In [2]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()

print("size.w is 21:              ", shop.products.count_documents({"size.w": 21}))
print("size is exactly {'w': 21}: ", shop.products.count_documents({"size": {"w": 21}}))
print("size.h is 34 as well:      ",
      shop.products.count_documents({"size.w": 21, "size.h": 34}))


size.w is 21:               9
size is exactly {'w': 21}:  0
size.h is 34 as well:       2


Two conditions on two paths, written as two keys, is the normal way to ask about a subdocument. It
also does not care about field order, does not break when somebody adds a third field to `size`, and
can use an index on either path.

Whole-document equality has exactly one good use: when the subdocument really is a single value with
a fixed shape, such as a coordinate pair you always write the same way. Even then it is fragile:


In [3]:
exact = shop.products.find_one({"_id": 0})["size"]

print("the subdocument:        ", exact)
print("matching it:            ", shop.products.count_documents({"size": exact}))
print("same values, keys swapped:",
      shop.products.count_documents({"size": {"h": exact["h"], "w": exact["w"]}}))
print("BSON keeps field order, and equality compares it")


the subdocument:         {'w': 21, 'h': 34}
matching it:             2
same values, keys swapped: 0
BSON keeps field order, and equality compares it


### Operators, and where they go

An operator is a key beginning with `$`, and it lives inside the field it applies to:


In [4]:
print("over 1900:   ", shop.products.count_documents({"price": {"$gt": 1900}}))
print("a range:     ", shop.products.count_documents({"price": {"$gte": 100, "$lte": 200}}))
print("not a laptop:", shop.products.count_documents({"kind": {"$ne": "laptop"}}))
print("one of three:",
      shop.products.count_documents({"kind": {"$in": ["laptop", "mouse", "cable"]}}))


over 1900:    19
a range:      30
not a laptop: 400
one of three: 300


Two operators in one dictionary are both applied, which is how a range is written. Separate fields
are combined with and, so `{"kind": "laptop", "price": {"$lt": 100}}` means both.

For or, there is `$or`, and it is one of the few operators that goes at the top level, because it is
about the document rather than about a field:


In [5]:
cheap_or_new = {"$or": [{"price": {"$lt": 50}}, {"tags": "new"}]}
print("cheap or new:", shop.products.count_documents(cheap_or_new))
print("cheap:       ", shop.products.count_documents({"price": {"$lt": 50}}))
print("new:         ", shop.products.count_documents({"tags": "new"}))


cheap or new: 214
cheap:        8
new:          212


### Arrays, and the condition that spreads

A query against an array field matches if **any element** matches. With one condition that is
exactly what you want:


In [6]:
print("tagged sale:", shop.products.count_documents({"tags": "sale"}))
print("tagged sale or new:", shop.products.count_documents({"tags": {"$in": ["sale", "new"]}}))
print("not tagged sale:   ", shop.products.count_documents({"tags": {"$nin": ["sale"]}}))


tagged sale: 220
tagged sale or new: 365
not tagged sale:    280


With two conditions it is not. Each condition is checked against the array independently, so they can
be satisfied by different elements:


In [7]:
print("the two documents:")
for document in shop.scores.find().sort("_id"):
    print("  ", document["_id"], document["entries"])

print()
print("kind a and n 5, as two conditions:",
      ids(shop.scores.find({"entries.kind": "a", "entries.n": 5})))


the two documents:
   1 [{'kind': 'a', 'n': 5}, {'kind': 'b', 'n': 1}]
   2 [{'kind': 'a', 'n': 1}, {'kind': 'b', 'n': 5}]

kind a and n 5, as two conditions: [1, 2]


Both documents. Document 2 has an element with `kind` `a` and a different element with `n` 5, and
the query never asked for them to be the same element.

`$elemMatch` asks for one element satisfying everything:


In [8]:
print("with $elemMatch:", ids(shop.scores.find({"entries": {"$elemMatch": {"kind": "a", "n": 5}}})))
print()
print("and it is only needed for two or more conditions:")
print("  one condition, plain:      ", ids(shop.scores.find({"entries.n": 5})))
print("  one condition, $elemMatch: ", ids(shop.scores.find({"entries": {"$elemMatch": {"n": 5}}})))


with $elemMatch: [1]

and it is only needed for two or more conditions:
  one condition, plain:       [1, 2]
  one condition, $elemMatch:  [1, 2]


The rule worth memorizing: **two or more conditions on the same array need `$elemMatch`**. One
condition does not, and wrapping it costs nothing but noise.

This is the failure that produces a plausible wrong answer rather than an error, which is why it is
worth a section of its own rather than a line in a table.

### Missing, null, and the difference

`{"v": None}` is not "has no value". It is "has no value **or** has the value null":


In [9]:
print("the three documents:", [document for document in shop.maybe.find().sort("_id")])
print()
print("v is None:              ", ids(shop.maybe.find({"v": None})))
print("v exists at all:        ", ids(shop.maybe.find({"v": {"$exists": True}})))
print("v is present and not null:", ids(shop.maybe.find({"v": {"$ne": None}})))
print("v is really null:       ", ids(shop.maybe.find({"v": {"$type": "null"}})))


the three documents: [{'_id': 1, 'v': None}, {'_id': 2}, {'_id': 3, 'v': 7}]

v is None:               [1, 2]
v exists at all:         [1, 3]
v is present and not null: [3]
v is really null:        [1]


Four questions, four different answers, and only two of them are the ones people usually mean.
`$exists` is about the field being there. `$type: "null"` is about it being there and being null.
`{"v": None}` is both at once, and it is the one everybody writes.

### When to reach for which

| What you want | How to write it |
|---|---|
| a field inside a subdocument | `{"size.w": 21}` |
| a subdocument, exactly and in order | `{"size": {"w": 21, "h": 34}}`, and think again |
| a comparison | `{"price": {"$gt": 100}}` |
| a range | `{"price": {"$gte": 100, "$lte": 200}}` |
| one of several values | `{"kind": {"$in": [...]}}` |
| none of several values | `{"kind": {"$nin": [...]}}` |
| either of two conditions | `{"$or": [{...}, {...}]}` |
| any element of an array | `{"tags": "sale"}`, with no operator |
| one element matching two conditions | `{"entries": {"$elemMatch": {...}}}` |
| the field to be present | `{"v": {"$exists": True}}` |
| the field to be present and not null | `{"v": {"$ne": None}}` |
| the field to be missing | `{"v": {"$exists": False}}` |

The default for anything nested is dot notation. Reach for whole-document equality only when the
subdocument is genuinely one value, and for `$elemMatch` the moment a second condition touches the
same array.

### A search function that means what it says, finished


In [10]:
def search(shop, kind=None, max_price=None, tags=(), min_width=None, must_have_stock=False):
    """Build a filter out of the parts that were asked for, each at the right level."""
    query = {}
    if kind is not None:
        query["kind"] = kind
    if max_price is not None:
        query["price"] = {"$lte": max_price}
    if tags:
        query["tags"] = {"$all": list(tags)}                        # every one of them, not any
    if min_width is not None:
        query["size.w"] = {"$gte": min_width}                       # a path, not a nested dict
    if must_have_stock:
        query["stock"] = {"$gt": 0}
    return query


for description, arguments in (
        ("laptops under 200", {"kind": "laptop", "max_price": 200}),
        ("anything tagged sale and new", {"tags": ["sale", "new"]}),
        ("wide monitors in stock", {"kind": "monitor", "min_width": 40,
                                    "must_have_stock": True}),
        ("everything", {})):
    query = search(shop, **arguments)
    print(f"  {description:30} {shop.products.count_documents(query):4}   {query}")


  laptops under 200                10   {'kind': 'laptop', 'price': {'$lte': 200}}
  anything tagged sale and new     67   {'tags': {'$all': ['sale', 'new']}}
  wide monitors in stock           35   {'kind': 'monitor', 'size.w': {'$gte': 40}, 'stock': {'$gt': 0}}
  everything                      500   {}


`$all` is the operator for "the array contains every one of these", as against `$in`, which is "any
of these". Getting those two the wrong way round is a quiet error of exactly the kind this notebook
is about, so the line says `$all` and the comment says what it means.

Every condition goes inside its field, the nested one is a path, and the whole thing is a plain
dictionary you can print, log, or paste into a shell.

### Where each part came from

| In `search` | What it relies on | The section that showed it |
|---|---|---|
| `{"price": {"$lte": ...}}` | an operator inside its field | Operators, and where they go |
| `{"size.w": {"$gte": ...}}` | dot notation into a subdocument | A nested dictionary is a value |
| `{"tags": {"$all": [...]}}` | an array condition over every element | Arrays, and the condition that spreads |
| `{"stock": {"$gt": 0}}` | a comparison rather than truthiness | Operators, and where they go |
| building a plain `dict` | a filter being data | The idea |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/05-query-operators-solutions.ipynb).

**1.** Count the products whose `size.h` is over thirty, two ways, and show only one works.


In [11]:
# your code here


**2.** Find the products priced between five hundred and six hundred.


In [12]:
# your code here


**3.** Count the products that are a mouse or a cable, with one filter.


In [13]:
# your code here


**4.** Show that two conditions on `entries` match both documents, and `$elemMatch` matches one.


In [14]:
# your code here


**5.** In `maybe`, find the documents where `v` is missing entirely.


In [15]:
# your code here


**6.** Count the products tagged both `sale` and `new`, then either.


In [16]:
# your code here


## Common errors

### pymongo.errors.OperationFailure: unknown top level operator: $gte


In [17]:
try:
    shop.products.count_documents({"$gte": 100})
except pymongo.errors.OperationFailure as error:
    print(failed(error).split(". If you")[0])


OperationFailure: unknown top level operator: $gte


The operator went where the field should be. `{"$gte": 100}` says "this document is at least 100",
which is not a thing a document can be.

The message goes on to mention `$getField`, which is worth explaining rather than skipping: MongoDB
is allowing for the possibility that you really do have a field called `$gte`, because a field name
may begin with a dollar sign. It almost never does.


In [18]:
print("what you meant:", shop.products.count_documents({"price": {"$gte": 100}}))


what you meant: 482


### No error: the nested dictionary that matches nothing


In [19]:
print("nested dictionary:", shop.products.count_documents({"size": {"w": 21}}))
print("dot notation:     ", shop.products.count_documents({"size.w": 21}))
print()
print("neither raised, and one of them is wrong")


nested dictionary: 0
dot notation:      9

neither raised, and one of them is wrong


Zero is a legitimate answer to a query, so nothing can tell you that you asked the wrong question.
In a report this shows up as a number that is too low, and it survives review because the filter
reads correctly in English.

The tell is a filter whose value is a dictionary with no `$` in it. That is always whole-document
equality, and it is almost always a mistake:


In [20]:
def looks_wrong(query):
    """A value that is a dict with no operators in it is whole-document equality."""
    return [field for field, value in query.items()
            if isinstance(value, dict) and not any(key.startswith("$") for key in value)]


print("suspicious:", looks_wrong({"size": {"w": 21}, "price": {"$gt": 5}}))
print("fine:      ", looks_wrong({"size.w": 21, "price": {"$gt": 5}}))


suspicious: ['size']
fine:       []


### No error: two conditions, two different elements


In [21]:
matched = ids(shop.scores.find({"entries.kind": "a", "entries.n": 5}))
print("matched:", matched)

for document in shop.scores.find({"_id": {"$in": matched}}).sort("_id"):
    satisfying = [entry for entry in document["entries"]
                  if entry["kind"] == "a" and entry["n"] == 5]
    print(f"  document {document['_id']}: elements satisfying both = {len(satisfying)}")


matched: [1, 2]
  document 1: elements satisfying both = 1
  document 2: elements satisfying both = 0


Document 2 matched a query that no element of it satisfies. That is not a bug and it is documented
behavior, and it is still the single most common way a MongoDB report is quietly wrong.

`$elemMatch` is the fix, and the habit is to reach for it whenever a second condition lands on an
array you have already filtered on:


In [22]:
print("with $elemMatch:", ids(shop.scores.find({"entries": {"$elemMatch": {"kind": "a", "n": 5}}})))


with $elemMatch: [1]


### No error: None matching the documents that have nothing


In [23]:
print("everything in maybe:", [document for document in shop.maybe.find().sort("_id")])
print()
print("v is None matched:", ids(shop.maybe.find({"v": None})),
      "<- document 2 has no v at all")


everything in maybe: [{'_id': 1, 'v': None}, {'_id': 2}, {'_id': 3, 'v': 7}]

v is None matched: [1, 2] <- document 2 has no v at all


A filter written to find "the ones where `v` was set to null" also returns every document that never
had a `v`. In a collection where the field was added later, that is most of them.

Say which you meant:


In [24]:
print("really null:     ", ids(shop.maybe.find({"v": {"$type": "null"}})))
print("really missing:  ", ids(shop.maybe.find({"v": {"$exists": False}})))
print("present, any value:", ids(shop.maybe.find({"v": {"$exists": True}})))


really null:      [1]
really missing:   [2]
present, any value: [1, 3]


In [25]:
client.close()
print("closed")


closed


## Recap

- A filter value that is a dictionary without operators is whole-document equality: same fields,
  same values, **same order**. `{"size": {"w": 21}}` almost never means what its author thought.
- Dot notation is the way into a subdocument or into the elements of an array: `"size.w"`,
  `"entries.n"`. It works in filters, projections, sorts and indexes.
- An operator lives inside the field it applies to. At the top level it is
  `unknown top level operator`, except for `$or`, `$and` and `$nor`, which really are about the
  whole document.
- A condition on an array matches if any element matches, so **two conditions can be satisfied by
  two different elements**. `$elemMatch` requires one element to satisfy all of them.
- `{"v": None}` matches both an explicit null and a missing field. `$exists` separates them and
  `$type: "null"` finds only the real nulls.
- `$in` is any of these, `$all` is every one of these, and `$nin` is none of these.


## What is next

**Update Operators** is the writing half of the same problem: `$set` against the `replace_one` that
silently dropped every field you left out, and the upsert that inserted a second copy instead of
updating the first.


---

&#8592; **Previous:** [find and find_one](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/04-find-and-find-one.ipynb)  &nbsp;·&nbsp;  [PyMongo and Beanie, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)
